# 17.1 數字個數相同，為什麼空間不同？


一張64列、32欄的權重表有2048個數字。要算它佔多少空間，還要知道每格怎麼存。**bit（位元）**有兩種狀態，八個bit組成一個**byte（位元組）**。FP32每格32bit，即4byte；FP16每格2byte；int8每格1byte。

因此同一張表的數值部分分別是8192、4096、2048byte。形狀和參數個數沒變，改的是能表示的數字範圍與細節。**量化**就是把原數值映到較少的離散選項，目標是減少表示成本，同時控制精度損失。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

weight = torch.zeros(64, 32)
for dtype in (torch.float32, torch.float16, torch.int8):
    converted = weight.to(dtype)
    count = converted.numel()
    per_value = converted.element_size()
    print(dtype, "元素", count, "每格bytes", per_value, "總bytes", count * per_value)

全零表只隔離空間問題，不是學好的模型。`to(dtype)`建立指定型別，`numel()`數元素，`element_size()`回報每格byte。三版都有2048格，最後一欄對上手算。

這裡尚未寫檔。完整檔案還有名稱、形狀、型別等附帶資訊，低位元版本也須保存還原刻度，所以檔案大小不能只用參數數量乘bit。運行時還有中間特徵、生成快取，訓練另有梯度和更新器紀錄；權重縮小不表示整臺機器佔用同比例縮小。

也不能直接用`.to(int8)`當一般量化配方：0.7可能變0，卻沒記下如何對應原範圍。全零在三種型別都精確，才適合本次空間比較。下一節補上這份對應規則。

練習把64列改128列，預測三種元素數都變4096，總byte加倍，每格byte不變，再執行核對。

<details>
<summary>補充：既有實驗、來源與完整測量範圍</summary>

現在也有[完整已訓練模型的量化實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/quantization.json)：141,568個模型參數的FP32數值合計566,272 bytes，保存成`fp32.pt`則是588,358 bytes。4-bit轉換後參數個數仍相同，數值儲存168,736 bytes，`model.pt`為182,277 bytes；8-bit則是226,336與242,085 bytes。三個檔案都不含optimizer（優化器，用來更新權重的工具）為下次更新保留的紀錄。原始浮點檔仍保存續訓用的隨機狀態：這是記錄隨機過程進行到哪裡的額外資料，讓之後的隨機操作可以接著進行，不是權重參數。兩種保存格式也含不同的metadata（附帶說明資料，例如訓練紀錄或量化轉換方式），因此檔案比不能直接當作純權重位元比。這是私有實跑產物的大小；另行公開、剔除隨機狀態後的匯出檔要重新量，不能照抄這些bytes。

4-bit／8-bit在這份實報中描述的是線性層加權係數的存法：115,200個係數換成低位元，其餘25,728個數字與640個偏置仍保存為FP32，共26,368個浮點數。偏置是每個輸出加權和後多加的數。量化版本還新增1,416個FP32刻度值，用來說明整數格子如何還原到原來的數值範圍；刻度的運算下一節再展開，本節先把它們的儲存也算進來。

| 數值儲存部分 | 原FP32模型（bytes） | 4-bit版本（bytes） | 8-bit版本（bytes） |
| --- | ---: | ---: | ---: |
| 115,200個加權係數 | 460,800 | 57,600 | 115,200 |
| 26,368個其餘數字與偏置，保留FP32 | 105,472 | 105,472 | 105,472 |
| 新增1,416個FP32還原刻度 | 0 | 5,664 | 5,664 |
| 合計 | 566,272 | 168,736 | 226,336 |

兩個4-bit係數放在一byte，所以115,200個需要115,200÷2＝57,600 bytes；保留的FP32數字占26,368×4＝105,472，刻度占1,416×4＝5,664。因此4-bit這一欄就是57,600＋105,472＋5,664＝168,736 bytes。這個合計已包括仍用FP32的數字與新增刻度；檔案的名稱、形狀與其他容器資訊，才是再加到合計以外的部分。因此要核對壓縮率，先問哪些數字改了表示，再按各部分的存法相加。

</details>
